## Daily readiness dashboard
#### Loads objective and subjective wellness data alongside recent activity data to suggest training priorities for the day

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import HTML, display

from hll.config import PROCESSED_DIR

# Day to show. None means "latest day that has data". Set e.g. "2026-10-18" to look back.
SELECTED_DATE = None

In [ ]:
well = pd.read_csv(PROCESSED_DIR / "wellness.csv", parse_dates=["date"])
well = well.sort_values("date").set_index("date")

print("rows", len(well), "| range", well.index.min().date(), "to", well.index.max().date())

# Share of missing values per column: tells us which fields are usable.
well.isna().mean().round(2)
# Look back at previous 14 days 
well.tail(14)

In [ ]:
well = well.asfreq("D")          # a row for every calendar day, so windows count days
well["sleep_h"] = well["sleep_secs"] / 3600

In [ ]:
def window(series, end_day, n_days):
    """The last n_days of a series, up to and including end_day."""
    return series.loc[end_day - pd.Timedelta(days=n_days - 1): end_day]

hrv_last7 = window(well["hrv"], day, 7)
hrv_last60 = window(well["hrv"], day, 60)

hrv_recent = hrv_last7.mean()
hrv_baseline = hrv_last60.mean()
hrv_baseline_sd = hrv_last60.std()
hrv_cv = 100 * hrv_last7.std() / hrv_last7.mean()

# Typical CV: the 7-day CV worked out for every day, averaged over the last 60 days.
cv_by_day = 100 * well["hrv"].rolling(7).std() / well["hrv"].rolling(7).mean()
hrv_cv_baseline = window(cv_by_day, day, 60).mean()

rhr_recent = window(well["resting_hr"], day, 7).mean()
rhr_baseline = window(well["resting_hr"], day, 60).mean()

sleep_last_night = well.loc[day, "sleep_h"]
sleep_usual = window(well["sleep_h"], day - pd.Timedelta(days=1), 7).mean()   # the 7 nights before

In [ ]:
hrv_pct = 100 * (hrv_recent / hrv_baseline - 1)           # % vs baseline
hrv_z = (hrv_recent - hrv_baseline) / hrv_baseline_sd     # distance from baseline in SDs
cv_ratio = hrv_cv / hrv_cv_baseline                       # 1.0 = typical
rhr_diff = rhr_recent - rhr_baseline                      # bpm vs baseline
sleep_shortfall = sleep_usual - sleep_last_night          # hours less than usual

In [ ]:
# last_valid_index() returns the date of the last row that has an HRV value,
# so if today's data hasn't synced we show the most recent day that has it.
day = well["hrv"].last_valid_index() if SELECTED_DATE is None else pd.Timestamp(SELECTED_DATE)
row = well.loc[day]          # .loc picks one row by its date index
print("Showing:", day.date())

In [ ]:
# Thresholds: my starting heuristics, tune them after looking at your own data.
HRV_WATCH_SD, HRV_FLAG_SD = 0.5, 1.0        # SDs below baseline
CV_WATCH, CV_FLAG = 1.25, 1.5               # CV as a multiple of your typical CV
RHR_WATCH_BPM, RHR_FLAG_BPM = 2, 4          # bpm above baseline

GOOD, WARN, CRIT = "#0ca30c", "#fab219", "#d03b3b"   # status colours (green, amber, red)

def rate(badness, watch, flag):
    """badness: bigger = worse. Returns (label, colour, icon). Icon + label so colour isn't alone."""
    if pd.isna(badness):
        return "No data", MUTED, "–"
    if badness >= flag:
        return "Flag", CRIT, "✖"
    if badness >= watch:
        return "Watch", WARN, "▲"
    return "OK", GOOD, "●"

# Differences from baseline.
hrv_pct = 100 * (row["hrv_7d"] / row["hrv_base"] - 1)             # % vs baseline
hrv_z = (row["hrv_7d"] - row["hrv_base"]) / row["hrv_base_sd"]    # in baseline SDs
cv_ratio = row["hrv_cv7"] / row["hrv_cv_base"]                    # 1.0 = typical
rhr_diff = row["rhr_7d"] - row["rhr_base"]                        # bpm vs baseline

# HRV is worse when LOWER, so its badness is the negative of z. Only drops are flagged.
hrv_rating = rate(-hrv_z, HRV_WATCH_SD, HRV_FLAG_SD)
cv_rating = rate(cv_ratio, CV_WATCH, CV_FLAG)
rhr_rating = rate(rhr_diff, RHR_WATCH_BPM, RHR_FLAG_BPM)

In [ ]:
# Reuse the colours from notebook 01 (BG, CARD, INK, MUTED), defined in this notebook too.
BG, CARD = "#12141f", "#1a1d2e"
INK, MUTED = "#e8eaf2", "#8b90a7"

def show(value, decimals=0):
    """Number to text with fixed decimals; missing -> dash."""
    return "–" if pd.isna(value) else f"{value:.{decimals}f}"

def status_tile(label, value, unit, detail, rating):
    """A tile: big number, a line of detail, and a status with a coloured edge and icon."""
    name, colour, icon = rating
    return f"""
    <div style="flex:1; background:{CARD}; border-radius:8px; padding:16px 14px;
                border-left:6px solid {colour};">
      <div style="color:{MUTED}; font-size:13px;">{label}</div>
      <div style="color:{INK}; font-size:32px; font-weight:700;">{value}
        <span style="font-size:14px; color:{MUTED};">{unit}</span></div>
      <div style="color:{MUTED}; font-size:13px;">{detail}</div>
      <div style="color:{INK}; font-size:13px; font-weight:600; margin-top:8px;">
        <span style="color:{colour};">{icon}</span> {name}</div>
    </div>"""

readiness_tiles = (
    status_tile("HRV (rMSSD), 7-day mean", show(row["hrv_7d"]), "ms",
                f"{show(hrv_pct, 1)}% vs 60-day baseline ({show(row['hrv_base'])} ms)", hrv_rating)
    + status_tile("HRV variability, 7-day CV", show(row["hrv_cv7"], 1), "%",
                  f"baseline {show(row['hrv_cv_base'], 1)}% ({show(cv_ratio, 2)}×)", cv_rating)
    + status_tile("Resting HR, 7-day mean", show(row["rhr_7d"], 1), "bpm",
                  f"{show(rhr_diff, 1)} bpm vs baseline ({show(row['rhr_base'], 1)})", rhr_rating)
)


In [ ]:
# Compare last night with the average of the PREVIOUS 7 nights.
# shift(1) moves the series down a row, so last night is not part of its own average.
well["sleep_prev7"] = well["sleep_h"].shift(1).rolling(7, min_periods=4).mean()
row = well.loc[day]           # re-pick the row, now that the new column exists

def hours_mins(h):
    """7.5 -> '7h 30m'. Missing -> dash."""
    if pd.isna(h):
        return "–"
    mins = round(h * 60)
    hrs, mins = divmod(mins, 60)       # divmod gives quotient and remainder
    return f"{hrs}h {mins:02d}m"

# Shortfall vs your usual (positive = slept less than usual). Only shortfalls are flagged.
SLEEP_WATCH_H, SLEEP_FLAG_H = 0.75, 1.5          # heuristics, tune to taste
sleep_gap = row["sleep_prev7"] - row["sleep_h"]
sleep_rating = rate(sleep_gap, SLEEP_WATCH_H, SLEEP_FLAG_H)

sleep_tile = status_tile(
    "Sleep, last night", hours_mins(row["sleep_h"]), "",
    f"{show(-sleep_gap, 1)} h vs 7-night average ({hours_mins(row['sleep_prev7'])})",
    sleep_rating,
)

In [ ]:
act = pd.read_csv(PROCESSED_DIR / "activities.csv", parse_dates=["start_date"])
act["date"] = act["start_date"].dt.normalize()

# Total icu_load per calendar day.
daily_load = act.groupby("date")["icu_load"].sum()

# Rest days have no activities, so they are missing from daily_load.
# reindex() builds every day and fill_value=0 sets rest days to zero load.
# The window ends the day BEFORE the selected morning: today's training hasn't happened yet.
load_end = day - pd.Timedelta(days=1)
all_days = pd.date_range(daily_load.index.min(), load_end)
daily_load = daily_load.reindex(all_days, fill_value=0)

acute = daily_load.rolling(7).mean()        # average daily load, last 7 days
chronic = daily_load.rolling(28).mean()     # average daily load, last 28 days
ratio = acute / chronic                     # acute:chronic ratio

print("load up to", load_end.date(), "| acute", round(acute.iloc[-1]), "| chronic", round(chronic.iloc[-1]),
      "| ratio", round(ratio.iloc[-1], 2))

In [ ]:
AC_LOW, AC_HIGH, AC_FLAG = 0.8, 1.3, 1.5     # commonly quoted bands; contested, see Limitations

def rate_ratio(r):
    """Returns (label, colour, icon). Low is not 'bad', just under-loaded or tapering."""
    if pd.isna(r):
        return "No data", MUTED, "–"
    if r >= AC_FLAG:
        return "High", CRIT, "✖"
    if r > AC_HIGH:
        return "Elevated", WARN, "▲"
    if r < AC_LOW:
        return "Low", WARN, "▲"
    return "In range", GOOD, "●"

latest_ratio = ratio.iloc[-1]
load_tile = status_tile(
    "Acute:chronic load", show(latest_ratio, 2), "",
    f"acute {show(acute.iloc[-1])} · chronic {show(chronic.iloc[-1])} (avg daily load)",
    rate_ratio(latest_ratio),
)

In [ ]:
import plotly.graph_objects as go          # (move to your imports cell)

# --- Chart: acute vs chronic over the last 8 weeks ---
recent = pd.DataFrame({"acute": acute, "chronic": chronic}).tail(56)

fig = go.Figure()
fig.add_scatter(x=recent.index, y=recent["acute"], name="Acute (7-day avg)",
                line=dict(color="#3987e5", width=2))
fig.add_scatter(x=recent.index, y=recent["chronic"], name="Chronic (28-day avg)",
                line=dict(color="#d95926", width=2))
fig.update_layout(
    template="plotly_dark", height=320,
    paper_bgcolor=CARD, plot_bgcolor=CARD,                    # match the tile colour
    title=dict(text="Acute vs chronic load", font=dict(size=16, color=INK)),
    legend=dict(orientation="h", y=1.15, x=0),                # legend in one row above
    margin=dict(t=70, l=50, r=20, b=40),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor="#2a2e45", title_text="Avg daily load")

# --- Tiles ---
def tile_row(tiles_html):
    """Put tiles side by side."""
    return f'<div style="display:flex; gap:12px; margin-bottom:12px;">{tiles_html}</div>'

display(HTML(f"""
<div style="background:{BG}; padding:16px 16px 4px; border-radius:10px; font-family:sans-serif;">
  <div style="color:{INK}; font-size:18px; margin-bottom:12px;">Readiness · {day:%a %d %b %Y}</div>
  {tile_row(readiness_tiles)}
  {tile_row(sleep_tile + load_tile)}
</div>"""))
fig.show()